# 머신러닝 기초 3장 · Colab 실행용 노트북 (LightGBM, CatBoost, shap)

사이트 '머신러닝 기초' 3장(부스팅)에서 LightGBM, CatBoost, shap이 필요한 셀만 모은 노트북입니다. 사이트를 만든 컴퓨터에는 이 세 패키지가 없어서, Colab에서 한 번 돌려 나온 출력을 본문에 넣습니다.

**할 일 (네 가지)**

1. Colab(https://colab.research.google.com)에서 **파일 → 노트북 업로드**를 눌러 이 파일(ml03_colab.ipynb)을 엽니다.
2. 메뉴의 **런타임 → 모두 실행**을 누릅니다. 무료 Colab에서 10분 안팎 걸립니다. 맨 위 설치 셀 뒤에 '세션을 다시 시작하라(Restart session)'는 안내가 나오면 그 버튼을 누른 뒤 다시 **런타임 → 모두 실행**을 누릅니다.
3. 맨 아래 셀까지 끝나면 **파일 → 다운로드 → .ipynb 다운로드**로 파일을 받습니다(셀의 출력이 파일 안에 함께 저장됩니다).
4. 받은 파일을 구글 드라이브의 **'사회약학 연구실'** 폴더에 올립니다.

중간에 빨간 오류가 나면 거기서 멈춘 상태 그대로 3, 4를 하면 됩니다(어느 셀에서 왜 멈췄는지가 파일에 남습니다). 셀 위의 제목에 적힌 '3장 ○ 절 셀 n'은 사이트 본문에서 그 출력이 들어갈 셀 번호입니다. 셀의 코드는 고치지 않고 그대로 실행해 주세요.

## 설치와 버전

In [ ]:
!pip install optuna lightgbm catboost shap

In [ ]:
import os, sys
import numpy, pandas, sklearn, xgboost, lightgbm, catboost, shap, optuna
print("python", sys.version.split()[0], " CPU", os.cpu_count())
for m in [numpy, pandas, sklearn, xgboost, lightgbm, catboost, shap, optuna]:
    print(m.__name__, m.__version__)

## 준비

본문 셀 2의 코드 그대로입니다. 자료를 사이트 주소에서 읽고 0장과 같은 분할과 전처리를 만듭니다.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_val_score, cross_validate)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score, log_loss

BASE = "https://socialp-ajou.tecentriq12.workers.dev/data/"
UA = {"User-Agent": "Mozilla/5.0"}   # 사이트가 파이썬 기본 요청을 막아 브라우저처럼 보이게 함
df = pd.read_csv(BASE + "ml_claims.csv", storage_options=UA)
y = df["admit_2023"]
X = df.drop(columns=["id", "admit_2023", "cost_2023"])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=2026)
cat_cols = ["insurance", "region", "smoking", "alcohol"]
num_cols = [c for c in X.columns if c not in cat_cols]

def cat_steps():
    """글자 열: 결측을 'missing' 범주로 채운 뒤 원-핫 (0-2장과 같음)"""
    return Pipeline([
        ("impute", SimpleImputer(strategy="constant",
                                 fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore",
                                 sparse_output=False))])

def make_pipe(model, num=num_cols, cat=cat_cols):
    """0장: 중앙값 대체 + 표준화 + 원-핫"""
    num_steps = Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())])
    prep = ColumnTransformer([("num", num_steps, num),
                              ("cat", cat_steps(), cat)])
    return Pipeline([("prep", prep), ("model", model)])

def make_tree_pipe(model, num=num_cols, cat=cat_cols):
    """1장: 중앙값 대체 + 원-핫 (표준화 없음)"""
    prep = ColumnTransformer(
        [("num", SimpleImputer(strategy="median"), num),
         ("cat", cat_steps(), cat)],
        verbose_feature_names_out=False)
    return Pipeline([("prep", prep), ("model", model)])

def make_nan_pipe(model, num=num_cols, cat=cat_cols):
    """3장: 숫자 열은 결측(NaN) 그대로, 글자 열만 원-핫"""
    prep = ColumnTransformer(
        [("num", "passthrough", num), ("cat", cat_steps(), cat)],
        verbose_feature_names_out=False).set_output(transform="pandas")
    return Pipeline([("prep", prep), ("model", model)])

def boot_auc(y_true, p, n_boot=1000, seed=2026):
    """1장 마 절: 시험 자료를 복원추출해 다시 구한 AUC n_boot개"""
    rng = np.random.default_rng(seed)
    yt, p, out = np.asarray(y_true), np.asarray(p), []
    for _ in range(n_boot):
        i = rng.integers(0, len(yt), len(yt))
        out.append(roc_auc_score(yt[i], p[i]))
    return np.array(out)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2026)
print(X_train.shape, X_test.shape, y_train.sum(), y_test.sum())

## 본문에 들어가지 않는 준비

아래 셀들이 쓰는 XGBoost 모형과 자료를 다시 만듭니다. 출력의 시험 AUC가 사이트의 값과 같거나 끝자리만 다르면 정상입니다(XGBoost 버전이 다르면 조금 다를 수 있습니다).

In [ ]:
# 이 셀은 본문에 들어가지 않습니다. 본문의 앞 셀(XGBoost 등)이 만든 것 가운데
# 아래 셀들이 쓰는 것만 같은 코드로 다시 만듭니다.
import xgboost as xgb
from xgboost import XGBClassifier
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

X_tr, X_val, y_tr, y_val = train_test_split(              # 본문 셀 4
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=2026)
prep_nan = make_nan_pipe(None).named_steps["prep"]        # 본문 셀 7
Z_train = prep_nan.fit_transform(X_train)
Z_test = prep_nan.transform(X_test)
Z_tr, Z_val = Z_train.loc[X_tr.index], Z_train.loc[X_val.index]   # 본문 셀 14

# 본문 셀 18에서 Optuna가 고른 조합 (사이트를 만든 환경, xgboost 3.4.2)
best_xgb = {"n_estimators": 200,
            "learning_rate": 0.08296006039772535,
            "max_depth": 1,
            "min_child_weight": 1.2474747561630017,
            "subsample": 0.6464876912292634,
            "colsample_bytree": 0.39626241159839404,
            "reg_lambda": 13.846707712074963,
            "reg_alpha": 0.0021251668855702768}
xgb_final = make_nan_pipe(XGBClassifier(random_state=2026, **best_xgb))
xgb_final.fit(X_train, y_train)                           # 본문 셀 21
p_xgb = xgb_final.predict_proba(X_test)[:, 1]
booster = xgb_final[-1].get_booster()                     # 본문 셀 23
contrib = booster.predict(xgb.DMatrix(Z_test), pred_contribs=True)   # 셀 24
who = int(np.argsort(-p_xgb)[75])                         # 본문 셀 25
print("XGBoost tuned, test AUC:", round(roc_auc_score(y_test, p_xgb), 4),
      " (site: 0.7843)")
print("patient who:", who, " (site: 2262)")

<!-- ml03_lgb_default -->
### 3장 다 절 셀 9에 들어갈 출력 · LightGBM 기본값과 잎을 줄인 설정

In [ ]:
import lightgbm as lgb
from lightgbm import LGBMClassifier
print("lightgbm", lgb.__version__)

few = dict(num_leaves=4, learning_rate=0.05, n_estimators=300,
           min_child_samples=50)
for name, params in [("default", {}), ("4 leaves", few)]:
    model = make_nan_pipe(LGBMClassifier(random_state=2026, verbose=-1,
                                         **params))
    t0 = time.perf_counter()
    auc = cross_val_score(model, X_train, y_train, cv=skf,
                          scoring="roc_auc")
    print(f"{name:9s} CV AUC {auc.mean():.4f} (SD {auc.std():.4f}),"
          f" {time.perf_counter() - t0:.1f} s")

<!-- ml03_lgb_leaves -->
### 3장 다 절 셀 10에 들어갈 출력 · num_leaves의 검증 곡선

In [ ]:
leaves = [2, 4, 8, 16, 31, 64]
rows = []
for nl in leaves:
    model = make_nan_pipe(LGBMClassifier(
        num_leaves=nl, learning_rate=0.05, n_estimators=300,
        random_state=2026, verbose=-1))
    r = cross_validate(model, X_train, y_train, cv=skf,
                       scoring="roc_auc", return_train_score=True)
    rows.append([nl, r["train_score"].mean(), r["test_score"].mean(),
                 r["test_score"].std()])
res_leaves = pd.DataFrame(rows, columns=["num_leaves", "train AUC",
                                         "CV AUC", "CV SD"])
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(leaves, res_leaves["train AUC"], "o-", label="training folds")
ax.plot(leaves, res_leaves["CV AUC"], "o-", label="cross-validation")
ax.set_xscale("log", base=2)
ax.set_xlabel("num_leaves (learning_rate 0.05, 300 trees)")
ax.set_ylabel("AUC")
ax.legend()
plt.tight_layout()
res_leaves.round(4)

<!-- ml03_lgb_cat -->
### 3장 다 절 셀 11에 들어갈 출력 · 범주형 열을 그대로 넘기기 (pandas category)

In [ ]:
Xl_train = X_train.astype({c: "category" for c in cat_cols})
print([str(t) for t in Xl_train[cat_cols].dtypes])
for name, model, data in [
        ("one-hot (make_nan_pipe)",
         make_nan_pipe(LGBMClassifier(random_state=2026, verbose=-1,
                                      **few)), X_train),
        ("category dtype, no pipeline",
         LGBMClassifier(random_state=2026, verbose=-1, **few), Xl_train)]:
    auc = cross_val_score(model, data, y_train, cv=skf,
                          scoring="roc_auc")
    print(f"{name:28s} CV AUC {auc.mean():.4f}")

<!-- ml03_cat -->
### 3장 라 절 셀 13에 들어갈 출력 · CatBoost: 범주형 열을 그대로, 기본값으로

In [ ]:
import catboost
from catboost import CatBoostClassifier
print("catboost", catboost.__version__)

Xk_train, Xk_test = X_train.copy(), X_test.copy()
Xk_train[cat_cols] = Xk_train[cat_cols].fillna("missing")  # 글자 열의 NaN은
Xk_test[cat_cols] = Xk_test[cat_cols].fillna("missing")    # 글자로 바꿔야 함

def cv_catboost(**params):
    """CatBoost의 5겹 교차검증 AUC (겹은 skf와 같음)"""
    aucs = []
    for tr, va in skf.split(Xk_train, y_train):
        m = CatBoostClassifier(cat_features=cat_cols, random_seed=2026,
                               verbose=0, **params)
        m.fit(Xk_train.iloc[tr], y_train.iloc[tr])
        p = m.predict_proba(Xk_train.iloc[va])[:, 1]
        aucs.append(roc_auc_score(y_train.iloc[va], p))
    return np.mean(aucs), np.std(aucs), m

for name, params in [("default", {}),
                     ("ordered", {"boosting_type": "Ordered"})]:
    t0 = time.perf_counter()
    mean, sd, m = cv_catboost(**params)
    if name == "default":
        cv_cat_default = mean
    used = m.get_all_params()
    print(f"{name:8s} CV AUC {mean:.4f} (SD {sd:.4f}),"
          f" {time.perf_counter() - t0:.0f} s")
    print("   ", {k: used[k] for k in ["boosting_type", "iterations",
                                       "learning_rate", "depth",
                                       "l2_leaf_reg"]})

<!-- ml03_lgb_es -->
### 3장 마 절 셀 15에 들어갈 출력 · LightGBM의 조기 종료 (callbacks)

In [ ]:
lgb_es = LGBMClassifier(n_estimators=2000, learning_rate=0.05,
                        num_leaves=4, random_state=2026, verbose=-1)
lgb_es.fit(Z_tr, y_tr, eval_set=[(Z_tr, y_tr), (Z_val, y_val)],
           eval_metric="binary_logloss",
           callbacks=[lgb.early_stopping(50, verbose=False),
                      lgb.log_evaluation(0)])
print("best_iteration_:", lgb_es.best_iteration_)
print({k: round(v["binary_logloss"], 4)
       for k, v in lgb_es.best_score_.items()})
p_lval = lgb_es.predict_proba(Z_val)[:, 1]
print("validation AUC:", round(roc_auc_score(y_val, p_lval), 4))
ax = lgb.plot_metric(lgb_es, metric="binary_logloss", figsize=(6.5, 3.4))

<!-- ml03_lgb_optuna -->
### 3장 마 절 셀 22에 들어갈 출력 · Optuna로 LightGBM 튜닝 (40번 시도)

In [ ]:
def lgb_objective(trial):
    params = dict(
        n_estimators=trial.suggest_int("n_estimators", 100, 1000,
                                       step=50),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3,
                                          log=True),
        num_leaves=trial.suggest_int("num_leaves", 2, 64, log=True),
        min_child_samples=trial.suggest_int("min_child_samples", 5, 200,
                                            log=True),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.3,
                                             1.0),
        reg_lambda=trial.suggest_float("reg_lambda", 0.01, 100,
                                       log=True),
        reg_alpha=trial.suggest_float("reg_alpha", 0.001, 10,
                                      log=True))
    model = make_nan_pipe(LGBMClassifier(
        subsample_freq=1, random_state=2026, verbose=-1, **params))
    return cross_val_score(model, X_train, y_train, cv=skf,
                           scoring="roc_auc").mean()

study_lgb = optuna.create_study(direction="maximize",
                                sampler=TPESampler(seed=2026))
t0 = time.perf_counter()
study_lgb.optimize(lgb_objective, n_trials=40)
print({k: round(v, 4) for k, v in study_lgb.best_params.items()})
print(round(study_lgb.best_value, 4), " trial",
      study_lgb.best_trial.number,
      f" ({time.perf_counter() - t0:.0f} s)")

<!-- ml03_colab_final -->
### 3장 바 절 셀 30에 들어갈 출력 · LightGBM과 CatBoost의 시험 AUC (비교 표에 더할 줄)

In [ ]:
lgb_final = make_nan_pipe(LGBMClassifier(
    subsample_freq=1, random_state=2026, verbose=-1,
    **study_lgb.best_params)).fit(X_train, y_train)
cat_final = CatBoostClassifier(cat_features=cat_cols, random_seed=2026,
                               verbose=0).fit(Xk_train, y_train)
b_xgb = boot_auc(y_test, p_xgb)
rows = []
for name, p, cv in [
        ("LightGBM tuned", lgb_final.predict_proba(X_test)[:, 1],
         study_lgb.best_value),
        ("CatBoost default", cat_final.predict_proba(Xk_test)[:, 1],
         cv_cat_default)]:
    b = boot_auc(y_test, p)
    lo, hi = np.percentile(b, [2.5, 97.5])
    d_lo, d_hi = np.percentile(b - b_xgb, [2.5, 97.5])
    rows.append([name, cv, roc_auc_score(y_test, p), lo, hi, d_lo, d_hi])
pd.DataFrame(rows, columns=["model", "CV AUC", "test AUC", "CI low",
                            "CI high", "vs XGB low", "vs XGB high"]
             ).round(3)

<!-- ml03_shap_pkg -->
### 3장 바 절 셀 29에 들어갈 출력 · shap 패키지로 같은 그림

In [ ]:
import shap
print("shap", shap.__version__)

explainer = shap.TreeExplainer(xgb_final[-1])
sv = explainer(Z_test)               # values, base_values, data
print(sv.values.shape, " largest difference from pred_contribs:",
      np.abs(sv.values - contrib[:, :-1]).max())
shap.plots.bar(sv, max_display=12)
shap.plots.beeswarm(sv, max_display=12)
shap.plots.waterfall(sv[who], max_display=10)